# Process Mining Milestone 1 - Polluter Script

Group C

In [1]:
import pandas as pd
import pm4py

In [3]:
log = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")
df = pm4py.convert_to_dataframe(log)

/Users/felixhauptmann/ProM-Assignment-Group-C/.venv/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

## Pollution Script

### Unanchored Events

In [4]:
print(df["EventOrigin"].unique())
print(df.columns.tolist())

<StringArray>
['Application', 'Workflow', 'Offer']
Length: 3, dtype: str
['Action', 'org:resource', 'concept:name', 'EventOrigin', 'EventID', 'lifecycle:transition', 'time:timestamp', 'case:LoanGoal', 'case:ApplicationType', 'case:concept:name', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'Accepted', 'MonthlyCost', 'Selected', 'CreditScore', 'OfferedAmount', 'OfferID']


In [24]:
df_unanchored = df.copy()

# simulate different data sources (for unique values of EventOrigin)
mask_app = df_unanchored["EventOrigin"] == "Application"
mask_offer = df_unanchored["EventOrigin"] == "Offer"

# convert timestamp objects into strings
df_unanchored["time:timestamp"] = df_unanchored["time:timestamp"].astype("string")

# change the format of timestamp for all rows of a simulated data source
    # format: dd.mm.YYYY HH:MM:SS
df_unanchored.loc[mask_app, "time:timestamp"] = (
    pd.to_datetime(df_unanchored.loc[mask_app, "time:timestamp"], errors="coerce")
    .dt.strftime("%d.%m.%Y %H:%M:%S")
)

# format: YYYY-mm-dd'T'HH:MM:SS:MS'Z'
df_unanchored.loc[mask_offer, "time:timestamp"] = (
    pd.to_datetime(df_unanchored.loc[mask_offer, "time:timestamp"], errors="coerce")
    .dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
)

In [25]:
# output to show heterogenous timestamp formats of different simulated data sources (EventOrigin)
for origin in ["Application", "Workflow", "Offer"]:
    print("\n", origin)
    print(
        df_unanchored[df_unanchored["EventOrigin"] == origin]
        .sort_values("time:timestamp")
        .head(5)[["EventOrigin", "time:timestamp"]]
    )


 Application
    EventOrigin       time:timestamp
0   Application  01.01.2016 09:51:15
1   Application  01.01.2016 09:51:15
5   Application  01.01.2016 09:52:36
41  Application  01.01.2016 10:16:11
40  Application  01.01.2016 10:16:11

 Workflow
   EventOrigin                    time:timestamp
2     Workflow  2016-01-01 09:51:15.774000+00:00
3     Workflow  2016-01-01 09:52:36.392000+00:00
4     Workflow  2016-01-01 09:52:36.403000+00:00
42    Workflow  2016-01-01 10:16:11.740000+00:00
43    Workflow  2016-01-01 10:17:31.573000+00:00

 Offer
    EventOrigin               time:timestamp
917       Offer  2016-01-02T09:17:05.720000Z
918       Offer  2016-01-02T09:17:08.762000Z
919       Offer  2016-01-02T09:19:21.330000Z
924       Offer  2016-01-02T09:21:26.034000Z
925       Offer  2016-01-02T09:21:42.022000Z


In [26]:
#check how many events per time format
s = df_unanchored["time:timestamp"].astype(str).str.strip()

iso_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}", na=False)
eu_mask = s.str.match(r"^\d{1,2}\.\d{1,2}\.\d{4}", na=False)
us_mask = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}", na=False)
utc_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$", na=False)

print("ISO:", iso_mask.sum())
print("European:", eu_mask.sum())
print("US-like:", us_mask.sum())
print("UTC:", utc_mask.sum())
print("unknown:", (~(iso_mask | eu_mask | us_mask)).sum())

ISO: 962480
European: 239373
US-like: 0
UTC: 193657
unknown: 414


In [27]:
df_unanchored.to_csv("../data/unanchored_events.csv.gz", index=False, sep=";", compression="gzip")

### Polluted Labels

In [10]:
import random
import numpy as np

In [11]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [12]:
def pollute_labels(df, rate = 0.001):
    df = df.copy()
    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    for index in indices:
        activity = df.loc[index, "concept:name"]
        case_id = df.loc[index, "case:concept:name"]
        df.loc[index, "concept:name"] = f"{activity} - Incident No. {case_id}"

    return df

In [13]:
df_polluted = df.copy()
df_polluted = pollute_labels(df_polluted)

In [14]:
print(f"unique cases df: {df["concept:name"].nunique()}")
print(f"unique cases polluted: {df_polluted["concept:name"].nunique()}")

unique cases df: 26
unique cases polluted: 1222


In [15]:
df_polluted["concept:name"].unique().tolist()

['A_Create Application',
 'A_Submitted',
 'W_Handle leads',
 'W_Complete application',
 'A_Concept',
 'A_Accepted',
 'O_Create Offer',
 'O_Created',
 'O_Sent (mail and online)',
 'W_Call after offers',
 'A_Complete',
 'W_Validate application',
 'A_Validating',
 'O_Returned',
 'W_Call incomplete files',
 'A_Incomplete',
 'W_Call incomplete files - Incident No. Application_652823628',
 'O_Accepted',
 'A_Pending',
 'A_Denied',
 'O_Refused',
 'O_Cancelled',
 'A_Cancelled',
 'O_Sent (online only)',
 'W_Call incomplete files - Incident No. Application_821425679',
 'W_Handle leads - Incident No. Application_1461440233',
 'W_Validate application - Incident No. Application_486645986',
 'W_Call after offers - Incident No. Application_909491524',
 'A_Validating - Incident No. Application_1101248362',
 'W_Call incomplete files - Incident No. Application_1868808301',
 'O_Sent (mail and online) - Incident No. Application_866541745',
 'A_Validating - Incident No. Application_190984886',
 'O_Accepted 

### Distorted Labels

In [16]:
df["concept:name"].unique()

<StringArray>
[      'A_Create Application',                'A_Submitted',
             'W_Handle leads',     'W_Complete application',
                  'A_Concept',                 'A_Accepted',
             'O_Create Offer',                  'O_Created',
   'O_Sent (mail and online)',        'W_Call after offers',
                 'A_Complete',     'W_Validate application',
               'A_Validating',                 'O_Returned',
    'W_Call incomplete files',               'A_Incomplete',
                 'O_Accepted',                  'A_Pending',
                   'A_Denied',                  'O_Refused',
                'O_Cancelled',                'A_Cancelled',
       'O_Sent (online only)',   'W_Assess potential fraud',
 'W_Personal Loan collection',    'W_Shortened completion ']
Length: 26, dtype: str

In [17]:
def distort_activity_label(label):
    replacements = {
        "application": "appl.",
        "Application": "App.",
        "offers": "ofrs.",
        "Offer": "Off.",
        "incomplete": "incompl.",
        "Incomplete": "Incompl.",
        "files": "docs.",
        "leads": "lds.",
        "Handle": "Hndl.",
        "Validate": "Valid.",
        "Validating": "Valid.",
        "Complete": "Comp.",
        "Created": "Crtd.",
        "Create": "Crt.",
        "Accepted": "Acc.",
        "Cancelled": "Canc.",
        "Submitted": "Subm.",
        "Pending": "Pend.",
        "Denied": "Den.",
        "Refused": "Ref.",
        "Returned": "Ret.",
        "potential": "pot.",
        "fraud": "frd.",
        "Shortened": "Short.",
        "completion": "compl.",
        "Personal": "Pers.",
        "Loan": "Ln.",
        "collection": "coll.",
        "online": "onl.",
        "mail": "email"
    }

    distorted = str(label)

    for original, replacement in replacements.items():
        if original in distorted:
            return distorted.replace(original, replacement, 1)

    return distorted + "."

In [18]:
def pollute_distorted_labels(
    df,
    activity_column="concept:name",
    rate=0.05,
    pollution_column="pollution_type"
):
    df = df.copy()

    if pollution_column not in df.columns:
        df[pollution_column] = None

    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    for index in indices:
        original_label = df.loc[index, activity_column]

        if pd.notna(original_label):
            distorted_label = distort_activity_label(original_label)

            df.loc[index, activity_column] = distorted_label
            df.loc[index, pollution_column] = "distorted_label"

    return df

In [19]:
df_polluted = pollute_distorted_labels(
    df_polluted,
    activity_column="concept:name",
    rate=0.05
)

In [20]:
print("Original unique activities:", df["concept:name"].nunique())
print("Polluted unique activities:", df_polluted["concept:name"].nunique())

df_polluted["pollution_type"].value_counts()

Original unique activities: 26
Polluted unique activities: 1248


pollution_type
distorted_label    60113
Name: count, dtype: int64

In [21]:
df_polluted["concept:name"].unique().tolist()

['A_Create Application',
 'A_Subm.',
 'W_Handle leads',
 'W_Complete application',
 'A_Concept',
 'A_Accepted',
 'O_Create Off.',
 'O_Created',
 'O_Sent (mail and online)',
 'W_Call after offers',
 'A_Complete',
 'W_Validate application',
 'A_Validating',
 'O_Returned',
 'W_Call incomplete files',
 'A_Incompl.',
 'W_Call incomplete files - Incident No. Application_652823628',
 'W_Validate appl.',
 'O_Accepted',
 'A_Pending',
 'A_Submitted',
 'O_Create Offer',
 'W_Call after ofrs.',
 'A_Denied',
 'O_Refused',
 'O_Cancelled',
 'W_Complete appl.',
 'A_Incomplete',
 'A_Valid.',
 'W_Call incompl. files',
 'A_Canc.',
 'A_Create App.',
 'A_Cancelled',
 'A_Acc.',
 'O_Ret.',
 'W_Handle lds.',
 'A_Pend.',
 'O_Sent (mail and onl.)',
 'O_Sent (online only)',
 'O_Crtd.',
 'A_Concept.',
 'A_Comp.',
 'O_Sent (onl. only)',
 'O_Canc.',
 'W_Call incomplete files - Incident No. Application_821425679',
 'W_Handle leads - Incident No. Application_1461440233',
 'W_Validate application - Incident No. Applica

## Insert the missing pollution scripts here!!!
After that execute the last cells below.

In [22]:
event_log_polluted = pm4py.convert_to_event_log(df_polluted)

In [23]:
pm4py.write_xes(event_log_polluted, "../data/noised.xes.gz")

exporting log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]